In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week6") \
    .master("local[*]") \
    .getOrCreate()

spark.range(5).show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/26 12:49:00 WARN Utils: Your hostname, vidhis-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.94.79.160 instead (on interface en0)
26/07/26 12:49:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/07/26 12:49:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 0:>        

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [5]:
import os

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Week6") \
    .master("local[*]") \
    .getOrCreate()

In [7]:
os.makedirs("data", exist_ok=True)

csv_data = """product_id,category,price,region,priority,amount
1,Electronics,1500,North,High,1500
2,Clothing,800,South,Low,800
3,Electronics,1200,North,Low,1200
4,Furniture,300,East,High,300
5,Electronics,2200,South,High,2200
"""

with open("data/source.csv", "w") as f:
    f.write(csv_data)

print("data/source.csv created")

data/source.csv created


In [8]:
df = spark.read.csv("data/source.csv", header=True, inferSchema=True)
df.printSchema()
df.show()


root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- amount: integer (nullable = true)

+----------+-----------+-----+------+--------+------+
|product_id|   category|price|region|priority|amount|
+----------+-----------+-----+------+--------+------+
|         1|Electronics| 1500| North|    High|  1500|
|         2|   Clothing|  800| South|     Low|   800|
|         3|Electronics| 1200| North|     Low|  1200|
|         4|  Furniture|  300|  East|    High|   300|
|         5|Electronics| 2200| South|    High|  2200|
+----------+-----------+-----+------+--------+------+



In [21]:
orders_data = [
    (101, "Completed", 1500),
    (102, "Pending", 800),
    (103, "Completed", 1200),
    (104, "Cancelled", 300),
    (105, "Completed", 900),
]
order_columns = ["order_id", "status", "amount"]

df_orders = spark.createDataFrame(orders_data, order_columns)
df_orders.show()
result = df_orders.filter((col("status") == "Completed") & (col("amount") > 1000))
result.show()

+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|     101|Completed|  1500|
|     102|  Pending|   800|
|     103|Completed|  1200|
|     104|Cancelled|   300|
|     105|Completed|   900|
+--------+---------+------+

+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|     101|Completed|  1500|
|     103|Completed|  1200|
+--------+---------+------+



In [10]:
df_revised = df.withColumnRenamed("category", "product_category") \
                .withColumn("price", col("price").cast("double"))
df_revised.printSchema()
df_revised.show()

root
 |-- product_id: integer (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- amount: integer (nullable = true)

+----------+----------------+------+------+--------+------+
|product_id|product_category| price|region|priority|amount|
+----------+----------------+------+------+--------+------+
|         1|     Electronics|1500.0| North|    High|  1500|
|         2|        Clothing| 800.0| South|     Low|   800|
|         3|     Electronics|1200.0| North|     Low|  1200|
|         4|       Furniture| 300.0|  East|    High|   300|
|         5|     Electronics|2200.0| South|    High|  2200|
+----------+----------------+------+------+--------+------+



In [12]:
df_with_tax = df.withColumn("final_price", col("amount") * 1.18)
df_with_tax.show()

+----------+-----------+-----+------+--------+------+-----------+
|product_id|   category|price|region|priority|amount|final_price|
+----------+-----------+-----+------+--------+------+-----------+
|         1|Electronics| 1500| North|    High|  1500|     1770.0|
|         2|   Clothing|  800| South|     Low|   800|      944.0|
|         3|Electronics| 1200| North|     Low|  1200|     1416.0|
|         4|  Furniture|  300|  East|    High|   300|      354.0|
|         5|Electronics| 2200| South|    High|  2200|     2596.0|
+----------+-----------+-----+------+--------+------+-----------+



In [15]:
from pyspark.sql import Row
parquet_data = [
    Row(user_id=1, name="John", amount=500),
    Row(user_id=2, name="Sara", amount=800),
    Row(user_id=None, name="Unknown", amount=200),   # will get filtered out
    Row(user_id=4, name="Amit", amount=1200),
]

df_parquet_source = spark.createDataFrame(parquet_data)
df_parquet_source.write.mode("overwrite").parquet("path/to/input")

print("Parquet source created")
df_parquet_source.show()

[Stage 8:=============================>                             (4 + 4) / 8]

Parquet source created
+-------+-------+------+
|user_id|   name|amount|
+-------+-------+------+
|      1|   John|   500|
|      2|   Sara|   800|
|   NULL|Unknown|   200|
|      4|   Amit|  1200|
+-------+-------+------+



In [16]:
df12 = spark.read.parquet("path/to/input")
df12_clean = df12.filter(col("user_id").isNotNull())

df12_clean.write.mode("overwrite").option("header", True).csv("path/to/output")

df12_clean.show()

+-------+----+------+
|user_id|name|amount|
+-------+----+------+
|      4|Amit|  1200|
|      1|John|   500|
|      2|Sara|   800|
+-------+----+------+



In [17]:
spark.read.csv("path/to/output", header=True, inferSchema=True).show()

+-------+----+------+
|user_id|name|amount|
+-------+----+------+
|      4|Amit|  1200|
|      1|John|   500|
|      2|Sara|   800|
+-------+----+------+



In [18]:
result = df.filter((col("region") == "North") | (col("priority") == "High"))
result.show()

+----------+-----------+-----+------+--------+------+
|product_id|   category|price|region|priority|amount|
+----------+-----------+-----+------+--------+------+
|         1|Electronics| 1500| North|    High|  1500|
|         3|Electronics| 1200| North|     Low|  1200|
|         4|  Furniture|  300|  East|    High|   300|
|         5|Electronics| 2200| South|    High|  2200|
+----------+-----------+-----+------+--------+------+

